# SQL Data Analysis 

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False

#### Overview

In [3]:
%%sql
SELECT COUNT(*) FROM TRIPS;

1 rows affected.

count
6938841


In [11]:
%%sql 
SELECT COUNT(*) FROM stations;

1 rows affected.

count
881


In [6]:
%%sql
SELECT DISTINCT COUNT(*) FROM bikes;

1 rows affected.

count
8085


#### Q. Which 10 stations had the most trips start there and which of those stations were busiest relative to their capacity?

In [ ]:
%%sql
select * from trips limit 10;

-- find for each station how many trips were made, the total count/sum of trips for that start_station

select start_station_id, COUNT(start_station_id) as total_trips from trips group by start_station_id order by total_trips desc limit 10;

-- what's the capacity for those stations? 

with station_trips as (
	select start_station_id, COUNT(start_station_id) as total_trips from trips group by start_station_id order by total_trips desc limit 10
)
select s.station_name, s.capacity, t.total_trips from station_trips t join stations s on t.start_station_id = s.station_id order by t.total_trips desc;

-- which stations experience the most departing trips relative to their capacity?
-- trips per capacity unit = total departing trips / capacity

In [4]:
with station_trips as (
	select start_station_id, COUNT(start_station_id) as total_trips from trips group by start_station_id order by total_trips desc limit 10
)
select s.station_name, s.capacity, t.total_trips, (t.total_trips/s.capacity) as trips_per_capacity from station_trips t join stations s on t.start_station_id = s.station_id order by t.total_trips desc;

10 rows affected.

10 rows affected.

10 rows affected.

10 rows affected.

station_name,capacity,total_trips,trips_per_capacity
Bay St / College St (East Side),31,52442,1691
York St / Queens Quay W,57,49422,867
Union Station,43,44546,1035
Bay St / Wellesley St W,35,42808,1223
College St / Major St,15,40156,2677
King St W / Portland St - SMART,31,37983,1225
Bay St / Queens Quay W (Ferry Terminal),35,37554,1072
Queen St W / John St,23,36895,1604
Baldwin St / Spadina Ave - SMART,20,35875,1793
College St / Huron St,19,34805,1831


Bay St / College St (East Side) had the highest departing trip volume at 52,442 trips. But College St/Major St experienced the highest usage intensity among the top 10 stations with 2,677 departing trips per capacity unit. 

#### Q. Which stations lost the most bikes over the year (more trips start than end there) and which gained the most? 

In [9]:
%%sql
with trips_start as (
select start_station_id, COUNT(start_station_id) as departures from trips group by 1
), 
trips_end as (
select end_station_id, COUNT(end_station_id) as arrivals from trips group by 1
),
flow as(
select COALESCE(s.start_station_id, e.end_station_id) as station_id, s.departures, e.arrivals, (coalesce(s.departures, 0) - coalesce(e.arrivals, 0)) as net_flow from trips_start s full outer join trips_end e on s.start_station_id = e.end_station_id 
)
select * from flow order by net_flow asc limit 10;
-- select * from flow order by net_flow desc limit 10;

10 rows affected.

station_id,departures,arrivals,net_flow
7015,28899,62034,-33135
7033,44546,56187,-11641
7047,29094,37214,-8120
7019,26901,33010,-6109
7076,49422,55213,-5791
7052,18623,23940,-5317
7203,24902,29483,-4581
7102,26702,31134,-4432
7048,12717,17037,-4320
7020,11943,16152,-4209


These stations have more arrivals than departures

In [10]:
%%sql
with trips_start as (
    select start_station_id, COUNT(start_station_id) as departures from trips group by 1
), 
trips_end as (
    select end_station_id, COUNT(end_station_id) as arrivals from trips group by 1
),
flow as(
    select COALESCE(s.start_station_id, e.end_station_id) as station_id, s.departures, e.arrivals, (coalesce(s.departures, 0) - coalesce(e.arrivals, 0)) as net_flow from trips_start s full outer join trips_end e on s.start_station_id = e.end_station_id 
)
select * from flow order by net_flow desc limit 10;

10 rows affected.

station_id,departures,arrivals,net_flow
7030,42808,27112,15696
7024,17190,7438,9752
7907,15321,7410,7911
7042,23427,16258,7169
7044,26179,19028,7151
7385,17304,10945,6359
7409,19956,13846,6110
7025,15333,9598,5735
7667,15203,9558,5645
7549,14702,9691,5011


These stations have more departures than arrivals

for this specific analysis, the data covers one year, so the results show overall station imbalances and does not reflect daily or seasonal changes in bike demand

#### Q. Top three stations with highest number of departing trips per month

In [15]:
%config SqlMagic.displaylimit = 0

In [16]:
%%sql
with cte1 as (
    select EXTRACT(month from start_time) as month, start_station_id as station_id, COUNT(start_station_id) as total_trip_count from trips group by station_id, month
),
cte2 as (
    select month, station_id , total_trip_count, ROW_NUMBER() over (partition by month order by total_trip_count desc) as rn from cte1
)
select cte2.month, cte2.station_id , s.station_name,  cte2.total_trip_count from cte2 join stations s on cte2.station_id = s.station_id where cte2.rn in (1, 2, 3);

36 rows affected.

month,station_id,station_name,total_trip_count
1,7006,Bay St / College St (East Side),1952
1,7033,Union Station,1899
1,7030,Bay St / Wellesley St W,1761
2,7006,Bay St / College St (East Side),2512
2,7033,Union Station,2336
2,7030,Bay St / Wellesley St W,2261
3,7006,Bay St / College St (East Side),2749
3,7030,Bay St / Wellesley St W,2573
3,7033,Union Station,2328
4,7006,Bay St / College St (East Side),3085


Bay St / College St (East Side) and York St / Queens Quay W are among the busiest departure stations throughout the year